In [1]:
# Step 1 — Import Libraries
# Description: Load required Python libraries for data analysis and CSV handling.

import pandas as pd
import numpy as np

In [2]:
# Step 2 — Load Dataset
# Description: Reads the EPC dataset containing archetypes and building attributes.

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR.csv"

df = pd.read_csv(file_path)

# Preview dataset
df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,PV_NPV_FLEX,BATTERY_NPV_FLEX,WINDOWS_IRR_FLEX,ROOF_IRR_FLEX,WALLS_IRR_FLEX,FAB_IRR_FLEX,HP_IRR_FLEX,SOLAR_THERMAL_IRR_FLEX,PV_IRR_FLEX,BATTERY_IRR_FLEX
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,0.000000,-4836.000000,-0.023127,-0.037221,0.036246,0.019304,0.135966,0.111072,NaN,NaN
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,1962.148428,-2562.700903,-0.015884,-0.074175,-0.017873,-0.009717,0.418858,0.135407,0.081951,-0.037591
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0555,-4.2237,100,170,...,-2000.000000,-4836.000000,0.115626,1.741603,NaN,NaN,NaN,-0.043296,NaN,NaN
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,1179.292669,-2385.903238,-0.079761,-0.058603,-0.003530,0.003958,-0.012701,0.005398,0.073403,-0.029204
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.0547,-4.2283,150,120,...,-21000.000000,-4836.000000,0.372136,2.422694,NaN,NaN,NaN,0.139955,NaN,NaN


In [3]:
# Step 3 — Verify Required Columns
# Description: Confirm that required variables exist in the dataset.

required_columns = [
    "ARCHETYPE",
    "TOTAL_FLOOR_AREA",
    "FOOTPRINT_AREA",
    "FLOOR_HEIGHT"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if len(missing_columns) > 0:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")

All required columns are present.


In [4]:
# Step 4 — Clean Numeric Variables
# Description: Convert floor area and height variables to numeric format.

numeric_columns = [
    "TOTAL_FLOOR_AREA",
    "FOOTPRINT_AREA",
    "FLOOR_HEIGHT"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [5]:
# Step 5 — Calculate Building Counts per Archetype
# Description: Determine how many buildings belong to each archetype.

archetype_counts = df.groupby("ARCHETYPE").size().reset_index(name="BUILDING_COUNT")

archetype_counts

,ARCHETYPE,BUILDING_COUNT
0,Detached_1F,11
1,Detached_1F_1,1
2,Detached_2F,32
3,Detached_2F_1,20
4,Detached_2F_2,6
5,Detached_2F_3,1
6,Detached_2F_4,2
7,Detached_3F,1
8,EndTerrace_1F,1
9,EndTerrace_2F,2


In [6]:
# Step 6 — Define Function to Calculate Statistics
# Description: Computes summary statistics for a given variable grouped by archetype.

def calculate_stats(dataframe, variable):

    stats = dataframe.groupby("ARCHETYPE")[variable].agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ]).reset_index()

    # Rename columns for clarity
    stats = stats.rename(columns={
        "mean": f"{variable}_MEAN",
        "median": f"{variable}_MEDIAN",
        "std": f"{variable}_STD",
        "min": f"{variable}_MIN",
        "max": f"{variable}_MAX"
    })

    # Calculate coefficient of variation
    stats[f"{variable}_CV"] = stats[f"{variable}_STD"] / stats[f"{variable}_MEAN"]

    return stats

In [7]:
# Step 7 — Calculate Statistics for Each Variable
# Description: Generate descriptive statistics for each archetype variable.

floor_area_stats = calculate_stats(df, "TOTAL_FLOOR_AREA")

footprint_stats = calculate_stats(df, "FOOTPRINT_AREA")

floor_height_stats = calculate_stats(df, "FLOOR_HEIGHT")

In [8]:
# Step 8 — Merge All Statistics into One Table
# Description: Combine archetype counts and variable statistics into one dataset.

archetype_stats = archetype_counts

archetype_stats = archetype_stats.merge(floor_area_stats, on="ARCHETYPE", how="left")
archetype_stats = archetype_stats.merge(footprint_stats, on="ARCHETYPE", how="left")
archetype_stats = archetype_stats.merge(floor_height_stats, on="ARCHETYPE", how="left")

# Preview results
archetype_stats

,ARCHETYPE,BUILDING_COUNT,TOTAL_FLOOR_AREA_MEAN,TOTAL_FLOOR_AREA_MEDIAN,TOTAL_FLOOR_AREA_STD,TOTAL_FLOOR_AREA_MIN,TOTAL_FLOOR_AREA_MAX,TOTAL_FLOOR_AREA_CV,FOOTPRINT_AREA_MEAN,FOOTPRINT_AREA_MEDIAN,FOOTPRINT_AREA_STD,FOOTPRINT_AREA_MIN,FOOTPRINT_AREA_MAX,FOOTPRINT_AREA_CV,FLOOR_HEIGHT_MEAN,FLOOR_HEIGHT_MEDIAN,FLOOR_HEIGHT_STD,FLOOR_HEIGHT_MIN,FLOOR_HEIGHT_MAX,FLOOR_HEIGHT_CV
0,Detached_1F,11,66.727273,64.0,16.462630,51,104,0.246715,38.090909,32.000000,22.525339,25.500000,104.000000,0.591357,2.350000,2.300,0.097057,2.20,2.50,0.041301
1,Detached_1F_1,1,92.000000,92.0,NaN,92,92,NaN,92.000000,92.000000,NaN,92.000000,92.000000,NaN,2.440000,2.440,NaN,2.44,2.44,NaN
2,Detached_2F,32,162.281250,151.5,47.797445,68,309,0.294535,81.140625,75.750000,23.898723,34.000000,154.500000,0.294535,2.539062,2.425,0.641628,2.10,6.00,0.252703
3,Detached_2F_1,20,165.600000,164.0,35.074957,113,247,0.211805,82.800000,82.000000,17.537479,56.500000,123.500000,0.211805,2.496500,2.440,0.181841,2.10,2.87,0.072838
4,Detached_2F_2,6,157.166667,157.5,48.860686,87,217,0.310885,78.583333,78.750000,24.430343,43.500000,108.500000,0.310885,2.448333,2.425,0.063061,2.39,2.53,0.025757
5,Detached_2F_3,1,191.000000,191.0,NaN,191,191,NaN,95.500000,95.500000,NaN,95.500000,95.500000,NaN,2.100000,2.100,NaN,2.10,2.10,NaN
6,Detached_2F_4,2,293.500000,293.5,185.969083,162,425,0.633625,146.750000,146.750000,92.984542,81.000000,212.500000,0.633625,2.510000,2.510,0.127279,2.42,2.60,0.050709
7,Detached_3F,1,382.000000,382.0,NaN,382,382,NaN,127.333333,127.333333,NaN,127.333333,127.333333,NaN,2.500000,2.500,NaN,2.50,2.50,NaN
8,EndTerrace_1F,1,85.000000,85.0,NaN,85,85,NaN,85.000000,85.000000,NaN,85.000000,85.000000,NaN,2.400000,2.400,NaN,2.40,2.40,NaN
9,EndTerrace_2F,2,68.000000,68.0,0.000000,68,68,0.000000,34.000000,34.000000,0.000000,34.000000,34.000000,0.000000,2.400000,2.400,0.000000,2.40,2.40,0.000000


In [9]:
# Step 9 — Sort Results by Building Count
# Description: Sort archetypes from most common to least common.

archetype_stats = archetype_stats.sort_values(
    by="BUILDING_COUNT",
    ascending=False
)

archetype_stats

,ARCHETYPE,BUILDING_COUNT,TOTAL_FLOOR_AREA_MEAN,TOTAL_FLOOR_AREA_MEDIAN,TOTAL_FLOOR_AREA_STD,TOTAL_FLOOR_AREA_MIN,TOTAL_FLOOR_AREA_MAX,TOTAL_FLOOR_AREA_CV,FOOTPRINT_AREA_MEAN,FOOTPRINT_AREA_MEDIAN,FOOTPRINT_AREA_STD,FOOTPRINT_AREA_MIN,FOOTPRINT_AREA_MAX,FOOTPRINT_AREA_CV,FLOOR_HEIGHT_MEAN,FLOOR_HEIGHT_MEDIAN,FLOOR_HEIGHT_STD,FLOOR_HEIGHT_MIN,FLOOR_HEIGHT_MAX,FLOOR_HEIGHT_CV
2,Detached_2F,32,162.281250,151.5,47.797445,68,309,0.294535,81.140625,75.750000,23.898723,34.000000,154.500000,0.294535,2.539062,2.425,0.641628,2.10,6.00,0.252703
15,SemiDetached_2F,22,83.227273,82.0,34.584516,42,203,0.415543,41.613636,41.000000,17.292258,21.000000,101.500000,0.415543,2.406818,2.400,0.067074,2.20,2.59,0.027868
3,Detached_2F_1,20,165.600000,164.0,35.074957,113,247,0.211805,82.800000,82.000000,17.537479,56.500000,123.500000,0.211805,2.496500,2.440,0.181841,2.10,2.87,0.072838
0,Detached_1F,11,66.727273,64.0,16.462630,51,104,0.246715,38.090909,32.000000,22.525339,25.500000,104.000000,0.591357,2.350000,2.300,0.097057,2.20,2.50,0.041301
11,Flat,9,63.222222,60.0,6.119187,58,75,0.096789,31.611111,30.000000,3.059593,29.000000,37.500000,0.096789,2.347778,2.320,0.063004,2.28,2.43,0.026835
4,Detached_2F_2,6,157.166667,157.5,48.860686,87,217,0.310885,78.583333,78.750000,24.430343,43.500000,108.500000,0.310885,2.448333,2.425,0.063061,2.39,2.53,0.025757
16,SemiDetached_2F_1,5,130.600000,129.0,19.844395,104,150,0.151948,65.300000,64.500000,9.922197,52.000000,75.000000,0.151948,2.356000,2.360,0.096333,2.20,2.46,0.040888
9,EndTerrace_2F,2,68.000000,68.0,0.000000,68,68,0.000000,34.000000,34.000000,0.000000,34.000000,34.000000,0.000000,2.400000,2.400,0.000000,2.40,2.40,0.000000
6,Detached_2F_4,2,293.500000,293.5,185.969083,162,425,0.633625,146.750000,146.750000,92.984542,81.000000,212.500000,0.633625,2.510000,2.510,0.127279,2.42,2.60,0.050709
7,Detached_3F,1,382.000000,382.0,NaN,382,382,NaN,127.333333,127.333333,NaN,127.333333,127.333333,NaN,2.500000,2.500,NaN,2.50,2.50,NaN


In [10]:
# Step 10 — Export Archetype Statistics to CSV
# Description: Save the archetype statistics table as a new CSV file.

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_archetype_statistics.csv"

archetype_stats.to_csv(output_path, index=False)

print("Archetype statistics exported to:")
print(output_path)

Archetype statistics exported to:
/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_archetype_statistics.csv
